<a href="https://colab.research.google.com/github/AICHUCKY/Ai-with-Chucky-Colab-Notebooks/blob/main/ACE_Step_1_5_Colab_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 ACE-Step v1.5 - Google Colab Notebook
### 🔴 Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)

**Notes:**
- This requires a GPU runtime (T4 is usually sufficient).
- The first run will take a few minutes to download the models (approx 10GB).
- **Wait for the link ending in `gradio.live` to appear at the bottom.**

In [ ]:
# @title 1. Clone Repository & Patch Dependencies
import os
import sys

# 1. Clone the Hugging Face Space
if not os.path.exists("Ace-Step-v1.5"):
    print("Cloning ACE-Step v1.5 repository...")
    !git clone https://huggingface.co/spaces/ACE-Step/Ace-Step-v1.5
else:
    print("Repository already exists.")

# 2. Force Change Directory
# This ensures subsequent commands run inside the folder
os.chdir("/content/Ace-Step-v1.5")
print(f"Working directory set to: {os.getcwd()}")

# 3. Patch requirements.txt
# The original file asks for torch>=2.9.1 (future version) for Linux, which fails install.
# We relax this to allow the current stable torch version.
print("Patching requirements.txt...")
!sed -i 's/torch>=2.9.1/torch/g' requirements.txt

# 4. Patch app.py for Public Access
# By default, share=False. We change it to share=True to get a public Gradio link.
print("Patching app.py to enable public sharing...")
!sed -i 's/share=False/share=True/g' app.py

print("✅ Setup phase 1 complete.")

In [ ]:
# @title 2. Install Dependencies
import os

# Ensure we are in the correct folder
if os.getcwd() != "/content/Ace-Step-v1.5":
    os.chdir("/content/Ace-Step-v1.5")

print("Installing Python dependencies... (This may take 3-5 minutes)")

# Install dependencies with CUDA support
# We skip strict version checks for some libs to avoid Colab conflicts
!pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121

# Install nano-vllm using the direct path to guarantee it installs correctly
!pip install ./acestep/third_parts/nano-vllm

# Force uninstall flash-attn to prevent PyTorch fallback crashes on T4 GPUs
!pip uninstall -y flash-attn

# Install system dependency for audio processing
!apt-get install -y ffmpeg

print("✅ Dependencies installed.")

In [ ]:
# @title 3. Download Models & Launch Interface
# This cell first downloads the heavy model files (safetensors) and then starts the app.
import os

if os.getcwd() != "/content/Ace-Step-v1.5":
    os.chdir("/content/Ace-Step-v1.5")

# --- LITE MODE CONFIG ---
os.environ["SERVICE_MODE_DIT_MODEL_2"] = "" # Disable 2nd model download/load
os.environ["SERVICE_MODE_BACKEND"] = "vllm"

# 1. CREATE DOWNLOAD SCRIPT
download_code = """
import os
import sys
import torch

print("Initializing download sequence...")
current_dir = os.getcwd()
sys.path.insert(0, os.path.join(current_dir, "acestep", "third_parts", "nano-vllm"))

from acestep.handler import AceStepHandler

# We force CPU offload to avoid filling GPU RAM just for downloading
handler = AceStepHandler(persistent_storage_path=os.path.join(current_dir, "data"))

print("Downloading ACE-Step v1.5 Unified Repo (This includes DiT and LLM)...")
config_path = "acestep-v15-turbo"

try:
    if hasattr(handler, '_ensure_model_downloaded'):
        handler._ensure_model_downloaded(current_dir, config_path)
        print("\\n✅ Download complete.")
    else:
        print("Standard initialization...")
        handler.initialize_service(current_dir, config_path, device='cpu', offload_to_cpu=True)
except Exception as e:
    print(f"\\nDownload process finished: {e}")
"""

with open("download_models.py", "w") as f:
    f.write(download_code)

# 2. RUN DOWNLOAD
print("⏳ Starting Model Download... (This prevents timeouts)")
!python download_models.py

# 3. NUCLEAR PATCH FOR HF CACHE & TRANSFORMERS
print("🔧 Applying Deep Patches to HF Cache and Transformers library...")
patch_script = """
import os
import glob
import transformers

# Patch ACE-Step Custom Code
cache_dir = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
files = glob.glob(os.path.join(cache_dir, "**", "modeling_acestep_v15_turbo.py"), recursive=True)

for f_path in files:
    with open(f_path, "r") as f:
        content = f.read()

    # Force eager over flash-attn
    content = content.replace("'kernels-community/flash-attn3'", "'eager'")
    content = content.replace('"kernels-community/flash-attn3"', '"eager"')
    content = content.replace("flash_attention_2", "eager")
    content = content.replace("'sdpa'", "'eager'")
    content = content.replace('"sdpa"', '"eager"')

    with open(f_path, "w") as f:
        f.write(content)
    print(f"✅ Patched ACE-Step model: {f_path}")

# Patch HuggingFace Qwen3 Transformer Code
transformers_dir = os.path.dirname(transformers.__file__)
qwen3_files = glob.glob(os.path.join(transformers_dir, "models", "qwen3", "modeling_qwen3.py"))

for f_path in qwen3_files:
    with open(f_path, "r") as f:
        content = f.read()

    # Fix the 2D vs 4D tensor slicing bug in eager attention
    target_str = "causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]"
    replacement_str = '''
        if attention_mask is not None and attention_mask.dim() == 2:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]'''

    if "unsqueeze(1).unsqueeze(2)" not in content:
        content = content.replace(target_str, replacement_str)
        with open(f_path, "w") as f:
            f.write(content)
        print(f"✅ Patched Qwen3 Transformer: {f_path}")
"""
with open("patch_nuclear.py", "w") as f:
    f.write(patch_script)

!python patch_nuclear.py

os.environ["USE_FLASH_ATTENTION"] = "0"
os.environ["FLASH_ATTENTION_FORCE_BUILD"] = "FALSE"

# 4. LAUNCH APP
print("="*60)
print("🚀 Models ready. Launching ACE-Step Interface...")
print("🔗 Click the public link ending in 'gradio.live' below once it appears.")
print("="*60)

!python app.py